# Preprocessing Notebook

This notebook prepares analysis-ready tables from the raw YouTube collection outputs.

**How to use:**
- Set the `PERSON` environment variable (`bodeoum_tv` or `mimiminu`) or edit the default below;
  `scripts/run_pipeline.sh` runs both cases.
- Rerun all cells to produce outputs under `02-preprocessing/outputs/<person>/`.


## 0) Imports and configuration
This cell sets the dataset name and all input/output paths.


In [10]:
from __future__ import annotations

import json
import os
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from tqdm import tqdm

# ---- CONFIG ----
PERSON = os.environ.get("PERSON", "bodeoum_tv")  # "bodeoum_tv" or "mimiminu"
LANG = "ko"

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent:
    if (REPO_ROOT / "data").exists() and (REPO_ROOT / "02-preprocessing").exists():
        break
    REPO_ROOT = REPO_ROOT.parent
DATA_ROOT = REPO_ROOT / "data"
INPUT_CSV = DATA_ROOT / "input" / f"{PERSON}.csv"
YOUTUBE_JSONL = DATA_ROOT / PERSON / f"{PERSON}_youtube.jsonl"
COMMENTS_DIR = DATA_ROOT / PERSON / "comments"
TRANSCRIPTS_DIR = DATA_ROOT / PERSON / "transcripts"

OUTPUT_DIR = REPO_ROOT / "02-preprocessing" / "outputs" / PERSON
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Using PERSON=", PERSON)
print("Input CSV:", INPUT_CSV)
print("YouTube JSONL:", YOUTUBE_JSONL)
print("Comments dir:", COMMENTS_DIR)
print("Transcripts dir:", TRANSCRIPTS_DIR)
print("Output dir:", OUTPUT_DIR)


Using PERSON= bodeoum_tv
Input CSV: cancel-culture-sentiment-analysis/data/input/bodeoum_tv.csv
YouTube JSONL: cancel-culture-sentiment-analysis/data/bodeoum_tv/bodeoum_tv_youtube.jsonl
Comments dir: cancel-culture-sentiment-analysis/data/bodeoum_tv/comments
Transcripts dir: cancel-culture-sentiment-analysis/data/bodeoum_tv/transcripts
Output dir: cancel-culture-sentiment-analysis/02-preprocessing/outputs/bodeoum_tv


## 1) Load window map (CSV)
This builds a `slot -> video_url -> video_id` mapping based on the CSV order.


In [11]:
def extract_video_id(url: str) -> str | None:
    if "watch?v=" in url:
        return url.split("watch?v=", 1)[1].split("&", 1)[0]
    if "youtu.be/" in url:
        return url.split("youtu.be/", 1)[1].split("?", 1)[0]
    return None

window_df = pd.read_csv(INPUT_CSV)
window_df["video_id"] = window_df["video"].astype(str).apply(extract_video_id)

slot_rows = []
for window_label in ("pre", "cancel", "post"):
    rows = window_df[window_df["window"].str.lower() == window_label]
    for i, (_, row) in enumerate(rows.iterrows(), start=1):
        slot = f"{window_label}-{i}"
        slot_rows.append({
            "slot": slot,
            "window": window_label,
            "video": row["video"],
            "video_id": row["video_id"],
        })

slot_df = pd.DataFrame(slot_rows)
slot_df.head()


,slot,window,video,video_id
0,pre-1,pre,https://www.youtube.com/watch?v=eBP5Ds6o2Aw,eBP5Ds6o2Aw
1,pre-2,pre,https://www.youtube.com/watch?v=FCrjawLVIWo,FCrjawLVIWo
2,pre-3,pre,https://www.youtube.com/watch?v=iRo0SSChhwk,iRo0SSChhwk
3,pre-4,pre,https://www.youtube.com/watch?v=L5yvqZfpR64,L5yvqZfpR64
4,pre-5,pre,https://www.youtube.com/watch?v=A1L9uEZ0xNE,A1L9uEZ0xNE


## 2) Extract video metadata (`videos.csv`)
Parses `<person>_youtube.jsonl` and keeps records where `record_type == "video"`.


In [12]:
video_rows = []
with open(YOUTUBE_JSONL, "r", encoding="utf-8") as handle:
    for line in handle:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        if record.get("record_type") != "video":
            continue
        video_rows.append({
            "video_id": record.get("video_id"),
            "title": record.get("title"),
            "video_url": record.get("video_url"),
            "create_time": record.get("create_time"),
            "view_count": record.get("view_count"),
            "like_count": record.get("like_count"),
            "comment_count": record.get("comment_count"),
            "window_label": record.get("window_label"),
        })

videos_df = pd.DataFrame(video_rows)
videos_df.to_csv(OUTPUT_DIR / "videos.csv", index=False)
videos_df.head()


,video_id,title,video_url,create_time,view_count,like_count,comment_count,window_label
0,eBP5Ds6o2Aw,대형견 앞에서 얼어버린 소형견...(대형견=강형욱) [퍼피교육],https://www.youtube.com/watch?v=eBP5Ds6o2Aw,2024-05-18T00:00:00+00:00,1058798,6511,11000,pre
1,FCrjawLVIWo,메롱하면 혓바닥이 보라색인 강아지ㅣ견종백과 차우차우 편,https://www.youtube.com/watch?v=FCrjawLVIWo,2024-05-11T00:00:00+00:00,672851,8603,646,pre
2,iRo0SSChhwk,유기견 출신 친구가 산책하는 법 [집에가자] EP.7,https://www.youtube.com/watch?v=iRo0SSChhwk,2024-05-07T00:00:00+00:00,122687,3263,162,pre
3,L5yvqZfpR64,강형욱에게 트라우마가 된 견종 | 견종백과 와이어 폭스테리어편,https://www.youtube.com/watch?v=L5yvqZfpR64,2024-05-04T00:00:00+00:00,428831,6947,275,pre
4,A1L9uEZ0xNE,"엄정화님, 도대체 어떻게 키우신 건가요. 강형욱이 사랑에 빠진 진돗개 등장[강형욱의...",https://www.youtube.com/watch?v=A1L9uEZ0xNE,2024-04-27T00:00:00+00:00,2126742,37492,1900,pre


## 3) Flatten comments (`comments.csv`)
Reads `comments/*.info.json` and extracts comment fields + window slot.


In [13]:
comment_rows = []
comment_files = sorted(COMMENTS_DIR.glob("*.info.json"))

for path in tqdm(comment_files, desc="comments"):
    slot = path.stem.replace(".info", "")
    window = slot.split("-")[0]
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)

    video_id = payload.get("id")
    comments = payload.get("comments", [])
    for c in comments:
        comment_rows.append({
            "comment_id": c.get("id"),
            "parent_id": c.get("parent"),
            "author": c.get("author"),
            "author_id": c.get("author_id"),
            "timestamp": c.get("timestamp"),
            "like_count": c.get("like_count"),
            "text": c.get("text"),
            "video_id": video_id,
            "window": window,
            "slot": slot,
        })

comments_df = pd.DataFrame(comment_rows)
comments_df.to_csv(OUTPUT_DIR / "comments.csv", index=False)
comments_df.head()


comments: 100%|██████████| 21/21 [00:03<00:00,  6.12it/s]


,comment_id,parent_id,author,author_id,timestamp,like_count,text,video_id,window,slot
0,cmt_138242,root,@user,user_059184,1776610800,0,급여처리는 좀 너무하긴 했네요,e5N9eXLcfag,cancel,cancel-1
1,cmt_087614,root,@user,user_040657,1776603600,0,자기네 할 일만하고 성실하게 한다면 CCTV를 신경쓸일이 아닐텐데ㅠㅠ,e5N9eXLcfag,cancel,cancel-1
2,cmt_054237,root,@user,user_004080,1776556800,1,회사에서 일 하고 회사 앱 쓰면 조심 해야지!! 그런 내용을 봤던 안 봤던 왜 회사...,e5N9eXLcfag,cancel,cancel-1
3,cmt_077040,root,@user_070971,user_070971,1776470400,0,중견 대기업도 스트레스인데 중소회사에서 뭘 바라는게 많냐,e5N9eXLcfag,cancel,cancel-1
4,cmt_128144,root,@user,user_072464,1776470400,1,요새 강형욱님 영상 많이 떠서 보다가 여기까지 왔네요.. 일년전 영상인데...힘든 ...,e5N9eXLcfag,cancel,cancel-1


## 4) Build reply edges (`edges.csv`)
Creates reply graph edges from `comments.csv`.


In [14]:
edges_df = comments_df[comments_df["parent_id"].notna() & (comments_df["parent_id"] != "root")].copy()
edges_df = edges_df.rename(columns={
    "comment_id": "source_comment_id",
    "parent_id": "target_comment_id",
})
edges_df = edges_df[[
    "source_comment_id",
    "target_comment_id",
    "author_id",
    "video_id",
    "window",
    "slot",
]]
edges_df.to_csv(OUTPUT_DIR / "edges.csv", index=False)
edges_df.head()


,source_comment_id,target_comment_id,author_id,video_id,window,slot
39,cmt_056135,cmt_056134,user_072653,e5N9eXLcfag,cancel,cancel-1
40,cmt_056136,cmt_056134,user_026968,e5N9eXLcfag,cancel,cancel-1
60,cmt_130614,cmt_130613,user_016159,e5N9eXLcfag,cancel,cancel-1
61,cmt_130615,cmt_130614,user_058073,e5N9eXLcfag,cancel,cancel-1
65,cmt_054078,cmt_054077,user_016209,e5N9eXLcfag,cancel,cancel-1


## 5) Parse transcripts (`transcripts.csv`)
Reads `transcripts/*.ko.txt` and attaches window/slot.


In [15]:
transcript_rows = []
for path in sorted(TRANSCRIPTS_DIR.glob(f"*.{LANG}.txt")):
    name = path.name
    # expected: pre-1.ko.txt / cancel-1.ko.txt / post-1.ko.txt
    slot = name.split(".")[0]
    window = slot.split("-")[0]
    text = path.read_text(encoding="utf-8")
    transcript_rows.append({
        "slot": slot,
        "window": window,
        "transcript_text": text,
    })

transcripts_df = pd.DataFrame(transcript_rows)
transcripts_df.to_csv(OUTPUT_DIR / "transcripts.csv", index=False)
transcripts_df.head()


,slot,window,transcript_text
0,cancel-1,cancel,안녕하세요. 훈련사 강영웅입니다.\n사실 여부를 따지기에 앞서서\n이런 소식으로 좀...


## 6) Text cleaning
Enhanced preprocessing for informal Korean social media text:
1. Remove zero-width characters and HTML entities
2. Strip embedded URLs
3. Preserve Hangul (jamo + syllables), emojis, punctuation, Latin, digits
4. Normalize whitespace
5. Apply spacing normalization for Korean word boundaries
6. Deduplicate spam (same text, same author, 60s window)

> **Note on what is actually saved (reproducibility).** The two cleaning cells below run in
> sequence. The enhanced cleaner (steps 1-5) is used only to **filter empty comments and
> deduplicate spam** (step 6). The following cell then **recomputes `text_clean` with a
> minimal cleaner** (strip URLs, collapse whitespace) on the surviving rows, overwriting the
> enhanced version. The released `comments.csv` and all sentiment scores therefore use the
> minimal `text_clean`; the Korean spacing normalization (step 5) does not reach the output.
> Both cells are kept exactly as run for the paper so the outputs regenerate byte-for-byte.


In [16]:
import re
import kss
from datetime import datetime

url_re = re.compile(r"https?://\S+")
whitespace_re = re.compile(r"\s+")
zw_re = re.compile(r"[\u200b-\u200f\ufeff\u180b-\u180f]")
html_entity_re = re.compile(r"&[a-zA-Z]+;|&#x[a-fA-F0-9]+;|#[0-9]+;")
char_whitelist_re = re.compile(r"[\u1100-\u11FF\u3130-\u318F\uAC00-\uD7FFa-zA-Z0-9.,!?~()\u2000-\u206F\u2100-\u214F\u2190-\u21FF\u2200-\u22FF\u2300-\u23FF\u2500-\u257F\u2600-\u26FF\u2700-\u27BF\U0001F300-\U0001F6FF\U0001F700-\U0001F9FF\U0001FA00-\U0001FAFF]+")

def clean_text(text: str | None) -> str:
    if not text:
        return ""
    text = zw_re.sub("", text)
    text = html_entity_re.sub("", text)
    text = url_re.sub("", text)
    text = " ".join(char_whitelist_re.findall(text))
    text = whitespace_re.sub(" ", text).strip()
    return text

def normalize_korean_spacing(text: str) -> str:
    if not text or len(text.strip()) < 2:
        return text
    try:
        return " ".join(kss.split_sentences(text, strip=True, backend="fast"))
    except:
        return text

def deduplicate_spam(df: pd.DataFrame, time_window: int = 60) -> pd.DataFrame:
    df = df.copy()
    df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
    df = df.sort_values(["author_id", "timestamp"])
    mask = ~(
        (df["text_clean"] == df["text_clean"].shift(1))
        & (df["author_id"] == df["author_id"].shift(1))
        & ((df["timestamp"] - df["timestamp"].shift(1)) <= time_window)
    )
    return df[mask]

print("Cleaning comments...")
comments_df["text_clean"] = comments_df["text"].astype(str).apply(clean_text)
print("Cleaning transcripts...")
transcripts_df["text_clean"] = transcripts_df["transcript_text"].astype(str).apply(clean_text)

comments_df = comments_df[comments_df["text_clean"].str.strip().astype(bool)]
print(f"Comments before dedup: {len(comments_df)}")
comments_df = deduplicate_spam(comments_df)
print(f"Comments after dedup: {len(comments_df)}")

print("Normalizing Korean spacing for comments...")
comments_df["text_clean"] = comments_df["text_clean"].apply(normalize_korean_spacing)

comments_df.to_csv(OUTPUT_DIR / "comments.csv", index=False)
transcripts_df.to_csv(OUTPUT_DIR / "transcripts.csv", index=False)

print("\nSample cleaned comments:")
comments_df[["text", "text_clean"]].head()


Cleaning comments...
Cleaning transcripts...
Comments before dedup: 129374
Comments after dedup: 128532
Normalizing Korean spacing for comments...

Sample cleaned comments:


,text,text_clean
58156,이정도면 좋은사람이다 고마해라 그리고 무고죄로 신고쳐먹기싫으면 자숙하고 반성해라,이정도면 좋은사람이다 고마해라 그리고 무고죄로 신고쳐먹기싫으면 자숙하고 반성해라
124361,탱글이 다시 온 거 맞아요\n저는 다시 태어난 동물 많이 보았습니다\n사람이든...,탱글이 다시 온 거 맞아요 저는 다시 태어난 동물 많이 보았습니다 사람이든 동물이든...
3542,무조건 응원해요 두분~감사합니다!❤,무조건 응원해요 두분~감사합니다!❤
28242,힘드셨을텐데 잘하셨어요,힘드셨을텐데 잘하셨어요
28251,힘내세요 두분!,힘내세요 두분!


In [17]:
import re

url_re = re.compile(r"https?://\S+")
whitespace_re = re.compile(r"\s+")

def clean_text(text: str | None) -> str:
    if not text:
        return ""
    text = url_re.sub("", text)
    text = whitespace_re.sub(" ", text).strip()
    return text

comments_df["text_clean"] = comments_df["text"].astype(str).apply(clean_text)
transcripts_df["text_clean"] = transcripts_df["transcript_text"].astype(str).apply(clean_text)

comments_df = comments_df[comments_df["text_clean"].str.strip().astype(bool)]

comments_df.to_csv(OUTPUT_DIR / "comments.csv", index=False)
transcripts_df.to_csv(OUTPUT_DIR / "transcripts.csv", index=False)

comments_df[["text", "text_clean"]].head()


,text,text_clean
58156,이정도면 좋은사람이다 고마해라 그리고 무고죄로 신고쳐먹기싫으면 자숙하고 반성해라,이정도면 좋은사람이다 고마해라 그리고 무고죄로 신고쳐먹기싫으면 자숙하고 반성해라
124361,탱글이 다시 온 거 맞아요\n저는 다시 태어난 동물 많이 보았습니다\n사람이든...,탱글이 다시 온 거 맞아요 저는 다시 태어난 동물 많이 보았습니다 사람이든 동물이든...
3542,무조건 응원해요 두분~감사합니다!❤,무조건 응원해요 두분~감사합니다!❤
28242,힘드셨을텐데 잘하셨어요,힘드셨을텐데 잘하셨어요
28251,힘내세요 두분!,힘내세요 두분!


## 7) Summary
Quick sanity checks on row counts and output paths.


In [18]:
print("videos.csv", len(videos_df))
print("comments.csv", len(comments_df))
print("edges.csv", len(edges_df))
print("transcripts.csv", len(transcripts_df))
print("Output dir:", OUTPUT_DIR)


videos.csv 21
comments.csv 128532
edges.csv 42740
transcripts.csv 1
Output dir: cancel-culture-sentiment-analysis/02-preprocessing/outputs/bodeoum_tv
